# Structured Outputs, Grounding & Localization — From Pixels to JSON

> **Orbit 5 (Multimodal)** · Notebook 7 of 22

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/07_structured_outputs_grounding_and_localization.ipynb)

## Learning Objectives

By the end of this notebook you will be able to:

- **Understand the grounding problem** — connecting language descriptions to specific pixel regions in an image, and why this bridge between text and space matters for production systems.
- **Extract structured JSON, tables, and key-value pairs** from images using vision-language models, with robust parsing and validation strategies.
- **Implement visual grounding** — asking *"where is X?"* and receiving bounding-box coordinates that can be drawn, verified, and scored.
- **Design prompt schemas** that enforce structured VLM outputs, comparing open-ended, format-specified, and schema-enforced prompting styles.
- **Evaluate grounding quality** using Intersection-over-Union (IoU) and extraction precision metrics, establishing quantitative baselines for downstream use.

## 1 · The Grounding Problem

Vision-language models can *describe* an image with remarkable fluency. They can tell you the scene
contains "a red sports car parked beside a blue house under a golden sun." But can they **point**?
Can a VLM say "the red car is at pixel coordinates [120, 340, 280, 410]"? This is the
**grounding problem**: connecting free-form language descriptions to concrete pixel regions.

Why does grounding matter? In production, knowing "there is a defect" is practically useless.
What operators need is "the defect is at bounding box [x, y, w, h] on tile row 14." Without
spatial coordinates, downstream automation — cropping, routing, robotic pick-and-place — cannot
act on the model's knowledge. Grounding transforms *understanding* into *actionable output*.

We can distinguish three levels of visual grounding, each progressively harder:

| Level | Task | Output |
|-------|------|--------|
| **Image-level** | Classification / captioning | "This is an outdoor scene" |
| **Region-level** | Object localisation | `[x1, y1, x2, y2]` bounding boxes |
| **Pixel-level** | Segmentation | Binary or instance masks |

Qwen2.5-VL supports region-level grounding natively. When prompted correctly it returns bounding
boxes in a normalised coordinate system (0–1000 for both axes), which we can rescale to actual
pixel dimensions. This mirrors what we explored for *text* LLMs in **systems/12 — Structured
Outputs**: the same principle of constraining model output to fit a schema, except now the input
modality is visual rather than textual.

The fundamental challenge is that VLMs are trained to **generate text**, not coordinates.
Grounding is therefore an *emergent* ability — it varies dramatically across models, prompting
strategies, and scene complexity. Our job in this notebook is to measure that variance, build
robust extraction pipelines around it, and know when to trust (or distrust) the output.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
!pip install -q "transformers>=4.57.0" accelerate bitsandbytes torch pillow qwen-vl-utils requests

import torch, json, time, re
from PIL import Image, ImageDraw, ImageFont
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
import pandas as pd
import numpy as np

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"✓ {MODEL_ID} loaded  |  VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# ── Inference helper ────────────────────────────────────────────────────
def ask_vlm(image, prompt, max_tokens=1024):
    """Send a single image + text prompt to the VLM and return the response."""
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text], images=[image], padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_tokens)
    trimmed = ids[0, inputs.input_ids.shape[1]:]
    return processor.decode(trimmed, skip_special_tokens=True).strip()

## 2 · Building a Test Scene

We start by creating a synthetic scene with clearly positioned, labelled objects whose bounding
boxes we know *exactly*. This lets us measure grounding accuracy quantitatively — any error
in the predicted coordinates is entirely the model's, not ambiguity in the data.

In [ ]:
def make_grounding_scene():
    """Create an 800x600 scene with five labelled objects."""
    img = Image.new("RGB", (800, 600), "#f0f0f0")
    draw = ImageDraw.Draw(img)
    objects = [
        {"name": "red_car",     "bbox": [50, 200, 220, 340],  "color": "red",       "shape": "rectangle"},
        {"name": "blue_house",  "bbox": [300, 100, 500, 350], "color": "royalblue", "shape": "rectangle"},
        {"name": "green_tree",  "bbox": [580, 150, 720, 380], "color": "green",     "shape": "rectangle"},
        {"name": "yellow_sun",  "bbox": [650, 20, 760, 130],  "color": "gold",      "shape": "ellipse"},
        {"name": "brown_fence", "bbox": [40, 380, 760, 420],  "color": "sienna",    "shape": "rectangle"},
    ]
    for obj in objects:
        x1, y1, x2, y2 = obj["bbox"]
        if obj["shape"] == "ellipse":
            draw.ellipse([x1, y1, x2, y2], fill=obj["color"], outline="black", width=2)
        else:
            draw.rectangle([x1, y1, x2, y2], fill=obj["color"], outline="black", width=2)
        cx = (x1 + x2) // 2 - 20
        draw.text((cx, y2 + 5), obj["name"].replace("_", " "), fill="black")
    ground_truth = {obj["name"]: obj["bbox"] for obj in objects}
    return img, ground_truth

scene, gt_boxes = make_grounding_scene()
print("Ground truth bounding boxes:")
for name, bbox in gt_boxes.items():
    print(f"  {name}: {bbox}")
scene

## 3 · Structured Output from VLMs

The simplest form of structured output is asking a VLM to produce **JSON**. Unlike text LLMs
that can leverage function calling or constrained-decoding toolchains, most VLMs today rely on
**prompt-based schema enforcement**: you describe the desired JSON format inside the prompt and
hope the model follows it.

In practice, VLMs are *unreliable* JSON generators. They may wrap output in Markdown fences,
add conversational preamble before the JSON, or produce subtly malformed structures (trailing
commas, unquoted keys). A production pipeline must therefore include:

1. **Schema specification** — show the model the *exact* shape you want, ideally with a filled
   example rather than just field names.
2. **Robust parsing** — wrap `json.loads()` in `try/except` and have fallback extraction
   strategies (e.g., regex for code-block fences, brace-matching).
3. **Validation** — after parsing, verify required keys exist and values fall within expected
   ranges.
4. **Retry** — if parsing fails, re-prompt with the raw output asking the model to *fix* it.

Two key prompting approaches dominate:
- Provide a **JSON schema** with type annotations and descriptions.
- Provide a **filled example** (few-shot) — generally more reliable for VLMs because the model
  can mimic surface structure even if it doesn't understand JSON Schema semantics.

Let's test both patterns on our synthetic scene.

In [ ]:
print("═══ Structured JSON Extraction ═══")

prompt = """Analyze this image and describe every object as JSON:
{
  "objects": [
    {"name": "...", "color": "...", "position": "left|center|right", "size": "small|medium|large"}
  ],
  "scene_description": "one sentence",
  "object_count": N
}
Output ONLY valid JSON."""

response = ask_vlm(scene, prompt)
print(response)

try:
    parsed = json.loads(response)
    print(f"\n✓ Valid JSON — {len(parsed.get('objects', []))} objects detected")
except json.JSONDecodeError:
    print("\n✗ Invalid JSON — attempting cleanup...")
    match = re.search(r'```(?:json)?\s*(.*?)```', response, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group(1))
            print(f"✓ Recovered valid JSON from code block — {len(parsed.get('objects', []))} objects")
        except Exception:
            print("✗ Recovery failed")

### 3.1 · Key-Value Extraction from Forms

A common production task is extracting labelled fields from scanned forms, invoices, or intake
sheets. We synthesise a simple patient intake form and ask the model to return structured
key-value pairs.

In [ ]:
def make_form_image():
    """Create a synthetic patient intake form."""
    img = Image.new("RGB", (600, 400), "white")
    draw = ImageDraw.Draw(img)
    draw.text((200, 15), "PATIENT INTAKE FORM", fill="black")
    draw.line([(30, 45), (570, 45)], fill="black", width=2)
    fields = [
        ("Name:", "Jane M. Rodriguez"),
        ("Date of Birth:", "03/15/1988"),
        ("Phone:", "(555) 234-5678"),
        ("Insurance ID:", "BC-90412-X"),
        ("Allergies:", "Penicillin, Latex"),
        ("Primary Complaint:", "Persistent lower back pain, 3 weeks"),
    ]
    y = 65
    for label, value in fields:
        draw.text((40, y), label, fill="gray")
        draw.text((220, y), value, fill="black")
        draw.line([(220, y + 18), (560, y + 18)], fill="lightgray")
        y += 40
    return img, dict(fields)

form, gt_fields = make_form_image()
form

In [ ]:
print("═══ Key-Value Extraction ═══")

prompt = """Extract all form fields from this image as JSON:
{"fields": [{"label": "...", "value": "..."}]}
Output ONLY valid JSON."""

kv_response = ask_vlm(form, prompt)
print(kv_response)

In [ ]:
print("═══ Extraction Accuracy ═══\n")

def evaluate_kv_extraction(raw_text, ground_truth):
    """Compare extracted key-value pairs against ground truth."""
    try:
        data = json.loads(raw_text)
    except json.JSONDecodeError:
        m = re.search(r"\{[\s\S]*\}", raw_text)
        if m:
            data = json.loads(m.group())
        else:
            return {"parsed": False}

    extracted = {f["label"].rstrip(":"): f["value"] for f in data.get("fields", [])}
    gt_clean  = {k.rstrip(":"): v for k, v in ground_truth.items()}

    correct, total = 0, len(gt_clean)
    for key, gt_val in gt_clean.items():
        pred = extracted.get(key, "")
        match = pred.strip().lower() == gt_val.strip().lower()
        symbol = "✓" if match else "✗"
        correct += int(match)
        print(f"  {symbol}  {key:20s}  GT: {gt_val:40s}  Pred: {pred}")
    print(f"\nAccuracy: {correct}/{total} = {correct/total:.0%}")
    return {"parsed": True, "accuracy": correct / total}

evaluate_kv_extraction(kv_response, gt_fields)

## 4 · Visual Grounding — "Where Is X?"

Visual grounding inverts the usual VLM flow. Instead of *describing* what is in the image, we
give the model a **text query** and ask it to **locate** the referred object by returning
bounding-box coordinates.

Qwen2.5-VL can output bounding boxes when prompted with the right format. Internally the model
uses a normalised coordinate system (0–1000 for both axes); we rescale to pixel dimensions.
The key prompt pattern is: *"Locate [object] and return its bounding box as `[x1, y1, x2, y2]`
in pixel coordinates."*

It is important to distinguish grounding from classical **object detection**:

| | Object Detection | Visual Grounding |
|---|---|---|
| Input | Image + fixed category list | Image + free-form text query |
| Output | All instances of known classes | Box for the *referred* object |
| Flexibility | Limited to trained categories | Open-vocabulary |

Applications include accessibility ("describe what is at this screen location"), robotics
("pick up the red block on the left"), and visual question answering with spatial evidence.

In [ ]:
print("═══ Visual Grounding ═══")

prompt = """Locate the following objects in this image and return their bounding boxes.
Use the format [x1, y1, x2, y2] where coordinates are pixel values (image is 800x600).
Objects to locate: red car, blue house, green tree, yellow sun

Return as JSON:
{"objects": [{"name": "...", "bbox": [x1, y1, x2, y2]}]}"""

grounding_response = ask_vlm(scene, prompt)
print(grounding_response)

In [ ]:
def draw_boxes(image, predictions, ground_truth=None):
    """Draw predicted (red) and ground-truth (green) boxes on image."""
    img = image.copy()
    draw = ImageDraw.Draw(img)
    for obj in predictions:
        bbox = obj["bbox"]
        draw.rectangle(bbox, outline="red", width=3)
        draw.text((bbox[0], bbox[1] - 15), f"pred: {obj['name']}", fill="red")
    if ground_truth:
        for name, bbox in ground_truth.items():
            draw.rectangle(bbox, outline="lime", width=2)
            draw.text((bbox[0], bbox[3] + 2), f"gt: {name}", fill="green")
    return img

try:
    pred_data = json.loads(grounding_response)
    vis = draw_boxes(scene, pred_data["objects"], gt_boxes)
except Exception:
    m = re.search(r"\{[\s\S]*\}", grounding_response)
    if m:
        try:
            pred_data = json.loads(m.group())
            vis = draw_boxes(scene, pred_data["objects"], gt_boxes)
        except Exception:
            print("Could not parse grounding output — showing ground truth only")
            vis = draw_boxes(scene, [], gt_boxes)
    else:
        print("Could not parse grounding output — showing ground truth only")
        vis = draw_boxes(scene, [], gt_boxes)
vis

## 5 · Evaluating Grounding — Intersection over Union (IoU)

The standard metric for bounding-box quality is **Intersection over Union**:

$$\text{IoU} = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$

An IoU of **1.0** means the predicted and ground-truth boxes are identical; **0.0** means no
overlap at all. By convention, a prediction with **IoU ≥ 0.5** is considered a *correct*
detection — this threshold originates from the PASCAL VOC benchmark and remains widely used.

For structured extraction (key-value pairs, JSON fields) we use a complementary metric:
**field-level accuracy** — the fraction of extracted values that exactly match the ground truth
after normalisation. Together, IoU and field accuracy give us a full picture of a VLM's
structured-output reliability.

In [ ]:
def compute_iou(box1, box2):
    """Compute Intersection over Union for two [x1, y1, x2, y2] boxes."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

print("IoU Examples:")
print(f"  Perfect overlap:  {compute_iou([0,0,100,100], [0,0,100,100]):.2f}")
print(f"  50% overlap:      {compute_iou([0,0,100,100], [50,0,150,100]):.2f}")
print(f"  No overlap:       {compute_iou([0,0,100,100], [200,200,300,300]):.2f}")
print(f"  Partial overlap:  {compute_iou([0,0,100,100], [25,25,125,125]):.2f}")

In [ ]:
print("═══ Grounding IoU Benchmark ═══\n")

name_map = {
    "red car": "red_car", "red_car": "red_car",
    "blue house": "blue_house", "blue_house": "blue_house",
    "green tree": "green_tree", "green_tree": "green_tree",
    "yellow sun": "yellow_sun", "yellow_sun": "yellow_sun",
}

try:
    pred_objs = pred_data.get("objects", [])
except NameError:
    pred_objs = []

rows = []
for obj in pred_objs:
    pred_name = obj["name"].lower().strip()
    gt_key = name_map.get(pred_name)
    if gt_key and gt_key in gt_boxes:
        iou = compute_iou(obj["bbox"], gt_boxes[gt_key])
        status = "✓" if iou >= 0.5 else "✗"
        rows.append({"object": pred_name, "IoU": round(iou, 3), "correct": status})
        print(f"  {status}  {pred_name:15s}  IoU = {iou:.3f}")
    else:
        print(f"  ?  {pred_name:15s}  no ground-truth match")

if rows:
    avg_iou = np.mean([r["IoU"] for r in rows])
    print(f"\nMean IoU: {avg_iou:.3f}")
    print(f"Correct (IoU≥0.5): {sum(1 for r in rows if r['correct']=='✓')}/{len(rows)}")
else:
    print("No predictions could be matched to ground truth.")

## 6 · Prompt Engineering for Structured Outputs

Not all prompts are created equal when it comes to structured extraction. We can rank prompting
strategies by how tightly they constrain the model's output:

1. **Open-ended** — "Describe this image." → free-form prose, no guaranteed structure.
2. **Format-specified** — "Return a JSON object with keys: name, color, bbox." → likely JSON,
   but the model may add preamble or deviate from the schema.
3. **Schema-enforced** — provide a complete JSON skeleton with type annotations, valid value
   enums, and an explicit "Output ONLY valid JSON" instruction.

In practice, schema-enforced prompts with a **filled example** yield the highest parsing success
rate. Providing one correct example lets the model mimic the surface pattern — indentation,
quoting style, key order — rather than interpreting an abstract schema.

When JSON parsing still fails, a robust retry strategy is:

1. Extract from Markdown code fences (`` ```json ... ``` ``).
2. Brace-match: find the first `{` and last `}` and attempt to parse that substring.
3. Re-prompt: send the raw output back with *"Fix this JSON — output ONLY valid JSON."*

This mirrors the text-LLM structured output pipeline from **systems/12**, but without the
luxury of constrained decoding or grammar-guided generation — features that are beginning to
appear in VLM serving frameworks but are not yet mainstream.

In [ ]:
print("═══ Open-ended vs Schema-enforced ═══\n")

prompt_open = "Describe everything in this image."

prompt_schema = """Analyze this image. Return ONLY a JSON object matching this schema:
{
  "scene_type": "string — indoor/outdoor/abstract",
  "objects": [{"name": "string", "color": "string", "bbox_approx": [x1,y1,x2,y2]}],
  "dominant_colors": ["string"],
  "text_visible": "boolean"
}"""

print("--- Open-ended ---")
r1 = ask_vlm(scene, prompt_open, max_tokens=300)
print(r1[:500])

print("\n--- Schema-enforced ---")
r2 = ask_vlm(scene, prompt_schema, max_tokens=500)
print(r2[:500])

In [ ]:
print("═══ Parsability Comparison ═══\n")

for label, text in [("Open-ended", r1), ("Schema-enforced", r2)]:
    try:
        json.loads(text)
        print(f"  {label:20s} → ✓ valid JSON")
    except json.JSONDecodeError:
        m = re.search(r"\{[\s\S]*\}", text)
        if m:
            try:
                json.loads(m.group())
                print(f"  {label:20s} → ✓ JSON recovered via brace extraction")
            except Exception:
                print(f"  {label:20s} → ✗ not valid JSON")
        else:
            print(f"  {label:20s} → ✗ not valid JSON")

### 6.1 · A Robust JSON Parser for VLM Output

In [ ]:
def parse_vlm_json(raw_text):
    """Try multiple strategies to extract valid JSON from VLM output."""
    # Strategy 1: direct parse
    try:
        return json.loads(raw_text), "direct"
    except json.JSONDecodeError:
        pass
    # Strategy 2: extract from code block
    match = re.search(r"```(?:json)?\s*([\s\S]*?)```", raw_text)
    if match:
        try:
            return json.loads(match.group(1)), "code_block"
        except json.JSONDecodeError:
            pass
    # Strategy 3: find first { to last }
    start = raw_text.find("{")
    end = raw_text.rfind("}")
    if start != -1 and end != -1:
        try:
            return json.loads(raw_text[start:end+1]), "brace_extraction"
        except json.JSONDecodeError:
            pass
    return None, "failed"

# Test the parser
test_outputs = [
    '{"name": "test"}',
    'Here is the JSON:\n```json\n{"name": "test"}\n```',
    'The result is {"name": "test"} as requested.',
    'This is not valid JSON at all',
]
print("Robust parser tests:")
for text in test_outputs:
    result, method = parse_vlm_json(text)
    print(f"  [{method:16s}] {text[:50]:50s} → {result}")

## 7 · Real-World Application — Receipt Parsing

Let's put our structured extraction pipeline to work on a more realistic task: parsing a
receipt. We create a synthetic receipt image, extract itemised line data plus totals, and
validate the results against ground truth. This end-to-end exercise combines schema-enforced
prompting with the robust JSON parser we built above.

In [ ]:
def make_receipt():
    """Create a synthetic grocery receipt."""
    img = Image.new("RGB", (400, 500), "white")
    draw = ImageDraw.Draw(img)
    draw.text((120, 10), "FRESH MART", fill="black")
    draw.text((110, 30), "123 Main Street", fill="gray")
    draw.line([(30, 55), (370, 55)], fill="black")
    items = [
        ("Organic Milk 1gal", 5.49),
        ("Sourdough Bread", 4.29),
        ("Avocado x3", 3.97),
        ("Chicken Breast 2lb", 11.98),
        ("Mixed Greens", 3.49),
    ]
    y = 70
    for name, price in items:
        draw.text((40, y), name, fill="black")
        draw.text((300, y), f"${price:.2f}", fill="black")
        y += 25
    y += 10
    draw.line([(30, y), (370, y)], fill="black")
    subtotal = sum(p for _, p in items)
    tax = round(subtotal * 0.08, 2)
    total = round(subtotal + tax, 2)
    y += 10
    draw.text((40, y), "Subtotal", fill="black")
    draw.text((300, y), f"${subtotal:.2f}", fill="black")
    y += 25
    draw.text((40, y), "Tax (8%)", fill="black")
    draw.text((300, y), f"${tax:.2f}", fill="black")
    y += 25
    draw.text((40, y), "TOTAL", fill="black")
    draw.text((300, y), f"${total:.2f}", fill="black")
    gt = {"store": "FRESH MART", "items": [{"name": n, "price": p} for n, p in items],
          "subtotal": subtotal, "tax": tax, "total": total}
    return img, gt

receipt, receipt_gt = make_receipt()
print(f"Ground truth total: ${receipt_gt['total']:.2f}")
receipt

In [ ]:
print("═══ Receipt Extraction ═══\n")

prompt = """Extract all data from this receipt as JSON:
{
  "store_name": "string",
  "items": [{"name": "string", "price": float}],
  "subtotal": float,
  "tax": float,
  "total": float
}
Output ONLY valid JSON."""

receipt_response = ask_vlm(receipt, prompt)
parsed_receipt, method = parse_vlm_json(receipt_response)
print(f"Parse method: {method}")

if parsed_receipt:
    print(f"\nStore:    {parsed_receipt.get('store_name', '?')}")
    print(f"Items:    {len(parsed_receipt.get('items', []))}")
    print(f"Total:    ${parsed_receipt.get('total', 0):.2f}")
    print(f"GT Total: ${receipt_gt['total']:.2f}")
    if abs(parsed_receipt.get("total", 0) - receipt_gt["total"]) < 0.01:
        print("✓ Total matches ground truth")
    else:
        print("✗ Total does not match ground truth")
else:
    print("✗ Failed to parse receipt data")
    print(receipt_response[:300])

In [ ]:
print("═══ Pipeline Summary ═══\n")

summary = pd.DataFrame([
    {"Task": "JSON scene extraction", "Prompt style": "Schema-enforced",
     "Notes": "Returns object list with properties"},
    {"Task": "Key-value form extraction", "Prompt style": "Schema-enforced",
     "Notes": "Label/value pairs from form images"},
    {"Task": "Visual grounding (bbox)", "Prompt style": "Coordinate-specific",
     "Notes": "Returns [x1,y1,x2,y2] per object"},
    {"Task": "Receipt parsing", "Prompt style": "Schema-enforced",
     "Notes": "End-to-end structured extraction"},
])
print(summary.to_string(index=False))

## 8 · Exercises

**Exercise 1 — Dense Grounding Scene.**
Create a more complex scene with **8+ objects** at various sizes and positions. Run the
grounding pipeline on each object and compute per-object IoU. At what point does grounding
quality degrade — small objects? overlapping objects? unusual shapes?

**Exercise 2 — Business Card Extractor.**
Design a JSON schema for extracting data from business cards: name, title, company, email,
and phone. Create three synthetic card images with different layouts and measure field-level
accuracy. Which fields are hardest for the model?

**Exercise 3 — Auto-Retry Pipeline.**
Implement an extraction pipeline that makes **up to 3 attempts** with different prompt
strategies when JSON parsing fails: (a) schema-enforced, (b) few-shot with filled example,
(c) ask the model to fix its own output. Log which strategy succeeds and at which attempt.

In [ ]:
# ── Exercise 1 starter ─────────────────────────────────────────────────
def make_dense_scene(n_objects=10):
    """Create a scene with many randomly placed coloured rectangles."""
    img = Image.new("RGB", (800, 600), "#f5f5f5")
    draw = ImageDraw.Draw(img)
    colours = ["red", "blue", "green", "orange", "purple",
               "cyan", "magenta", "brown", "olive", "teal"]
    rng = np.random.RandomState(42)
    gt = {}
    for i in range(n_objects):
        w = rng.randint(40, 140)
        h = rng.randint(40, 120)
        x1 = rng.randint(10, 790 - w)
        y1 = rng.randint(10, 590 - h)
        colour = colours[i % len(colours)]
        name = f"{colour}_obj{i}"
        draw.rectangle([x1, y1, x1+w, y1+h], fill=colour, outline="black")
        draw.text((x1+2, y1+2), str(i), fill="white")
        gt[name] = [x1, y1, x1+w, y1+h]
    return img, gt

dense_scene, dense_gt = make_dense_scene()
print(f"Created scene with {len(dense_gt)} objects")
# TODO: Run grounding pipeline on each object and compute IoU
dense_scene

In [ ]:
# ── Exercise 2 starter — Business card schema ─────────────────────────
business_card_schema = {
    "name": "string — full name",
    "title": "string — job title",
    "company": "string — company name",
    "email": "string — email address",
    "phone": "string — phone number",
}

def make_business_card(name, title, company, email, phone):
    img = Image.new("RGB", (500, 280), "white")
    draw = ImageDraw.Draw(img)
    draw.rectangle([10, 10, 490, 270], outline="navy", width=2)
    draw.text((30, 30), name, fill="black")
    draw.text((30, 60), title, fill="gray")
    draw.text((30, 100), company, fill="navy")
    draw.text((30, 150), email, fill="black")
    draw.text((30, 180), phone, fill="black")
    return img

card = make_business_card("Alice Chen", "Senior Engineer", "Acme Corp", "alice@acme.com", "(555) 867-5309")
# TODO: Extract fields using schema-enforced prompt and measure accuracy
card

In [ ]:
# ── Exercise 3 starter — Auto-retry pipeline ─────────────────────────
def extract_with_retry(image, task_description, schema, max_retries=3):
    """Try multiple prompt strategies to get valid JSON."""
    strategies = [
        f"Extract {task_description} as JSON matching:\n{json.dumps(schema, indent=2)}\nOutput ONLY valid JSON.",
        f"Here is an example output:\n{json.dumps(schema, indent=2)}\n\nNow extract {task_description} from this image in the same format.",
        None,  # placeholder for fix-up strategy
    ]

    last_raw = None
    for i, strategy in enumerate(strategies):
        if strategy is None and last_raw:
            strategy = f"The following is malformed JSON. Fix it and return ONLY valid JSON:\n{last_raw[:500]}"
        elif strategy is None:
            continue
        raw = ask_vlm(image, strategy)
        result, method = parse_vlm_json(raw)
        print(f"  Attempt {i+1}: parse={method}")
        if result is not None:
            return result, i + 1
        last_raw = raw
    return None, max_retries

print("Auto-retry pipeline ready — use extract_with_retry(image, desc, schema)")

## 9 · Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | VLMs can produce structured JSON output, but **always** wrap parsing in `try/except` with fallback strategies (code-block extraction, brace-matching, re-prompting). |
| 2 | Visual grounding — returning bounding boxes for referred objects — is an **emergent** VLM capability. Accuracy varies significantly by model size, prompt phrasing, and scene complexity. |
| 3 | **IoU ≥ 0.5** is the standard threshold for a "correct" bounding-box prediction, inherited from the PASCAL VOC benchmark. |
| 4 | Schema-enforced prompts with filled examples **dramatically** improve extraction consistency compared to open-ended or format-only prompts. |
| 5 | The robust JSON parser pattern — *try direct → code block → brace extraction* — handles the majority of VLM output quirks and belongs in every extraction pipeline. |

## References

1. **Qwen2.5-VL Technical Report** — Bai et al. (2025). *Qwen2.5-VL: Scaling Vision-Language Models for Perception and Understanding.* arXiv:2502.13923.
2. **Grounding DINO** — Liu et al. (2023). *Grounding DINO: Marrying DINO with Grounded Pre-Training for Open-Set Object Detection.* ECCV 2024.
3. **Kosmos-2** — Peng et al. (2023). *Kosmos-2: Grounding Multimodal Large Language Models to the World.* arXiv:2306.14824.
4. Castalia **systems/12** — Structured Outputs for Text LLMs.
5. Castalia **multimodal/05** — Image Prompting & Visual Reasoning.
6. Castalia **multimodal/06** — OCR, Layout & Table Extraction.

---

[← 06 OCR, Layout & Table Extraction](06_ocr_layout_and_table_extraction.ipynb) | [08 Small VLMs & Multi-Image Workflows →](08_small_vlms_and_multi_image_workflows.ipynb)